## Configure local inputs

Set the following environment variables to your files or folders before launching Jupyter: `MOUSEGPT_KEYPOINTS_JSON`, `MOUSEGPT_VIDEO`. Paths may include spaces. Output variables select destinations; input variables select existing session data. Review the analysis parameters in the cells before running on a new experiment.


In [ ]:
import os
required_paths = ['MOUSEGPT_KEYPOINTS_JSON', 'MOUSEGPT_VIDEO']
missing_paths = [name for name in required_paths if not os.environ.get(name)]
if missing_paths:
    raise ValueError("Set these environment variables before running: " + ", ".join(missing_paths))


In [ ]:
import json
import numpy as np
from tqdm import tqdm
import cv2

from scipy.signal import savgol_filter

import matplotlib.pyplot as plt

In [ ]:
import os
print('Loading keypoints json ……')

json_path = os.environ['MOUSEGPT_KEYPOINTS_JSON']
with open(json_path, 'r') as f:
    data = json.load(f)
    
frame_num = len(data)
print(f'total {frame_num} frames')

In [ ]:
kpts = []

for frame in tqdm(data, desc='loading kpts'):
    kpt = np.array(frame['instances'][0]['keypoints'])
    kpts.append(kpt)

kpts = np.array(kpts)
print(kpts.shape)

In [ ]:


def filter(kpts):
    # Apply the Savitzky-Golay filter to the 3D keypoint data
    window_length = 40
    poly_order = 7

    # Ensure window length is odd
    if window_length % 2 == 0:
        window_length += 1

    filtered_data_3d = np.empty_like(kpts)

    # Apply the filter for each dimension separately
    for point in range(kpts.shape[1]):
        for dim in range(2):
            filtered_data_3d[:, point, dim] = savgol_filter(kpts[:, point, dim], window_length, poly_order)
    return filtered_data_3d

filtered_kpts = filter(kpts)

print(filtered_kpts[0])

In [ ]:



head_direction = filtered_kpts[:,2,:] - filtered_kpts[:,3,:]
body_direction = filtered_kpts[:,3,:] - filtered_kpts[:,4,:]


dot_products = np.einsum('ij,ij->i', head_direction, body_direction)


norms1 = np.linalg.norm(head_direction, axis=1)
norms2 = np.linalg.norm(body_direction, axis=1)


cos_theta = dot_products / (norms1 * norms2)


cos_theta = np.clip(cos_theta, -1.0, 1.0)
angles = np.arccos(cos_theta)


cross_products = body_direction[:, 0] * head_direction[:, 1] - body_direction[:, 1] * head_direction[:, 0]


angles = np.where(cross_products < 0, -angles, angles)


angle_deg = np.degrees(angles)
    

angles[0], angle_deg.max(), angle_deg.min()

In [ ]:
from scipy.signal import butter, filtfilt


def lowpass_filter(data, cutoff, fs, order):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    y = filtfilt(b, a, data)
    return y


filtered_deg = lowpass_filter(angle_deg, cutoff=1, fs=120, order=3)

In [ ]:

fig, axs = plt.subplots(2,1)

lines = []

axs[0].axhline(0, color='r', linestyle='--')
axs[1].axhline(0, color='r', linestyle='--')
axs[0].set_xlabel('Frame')
axs[0].set_ylabel('angle')

limit = np.max(np.abs(angles))
axs[0].set_ylim(-limit, limit)

line, = axs[0].plot([], [], 'b-')
lines.append(line)

axs[1].set_xlabel('Frame')
axs[1].set_ylabel('degree')

limit = np.max(np.abs(angle_deg))
axs[1].set_ylim(-limit, limit)

line, = axs[1].plot([], [], 'b-')
lines.append(line)

In [ ]:
import os

cap = cv2.VideoCapture(os.environ['MOUSEGPT_VIDEO'])


fps = cap.get(cv2.CAP_PROP_FPS)


output_width = int(1680)
output_height = int(800)
output_fps = fps//2
output_filename = 'angle_filter_full_16.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_filename, fourcc, output_fps, (output_width, output_height))

In [ ]:
# index = 356000

# cap.set(cv2.CAP_PROP_POS_FRAMES, 356000)
index = 0

while cap.isOpened():
    index += 1
    print(index, end='\r')

    if index % 2 == 0:
        ret = cap.grab()
        continue
    else:
        ret, frame = cap.read()
    if not ret:
        break
    

    end_index = min(frame_num-1, index + 2)
    start_index = end_index-600
    speed_start_index = max(0, start_index)
    
    # Ear 1
    current_speed_data = np.zeros(600)
    current_speed_data[max(0, 600-(end_index-speed_start_index)):600] = angles[speed_start_index:end_index]

    axs[0].set_xlim(max(start_index, 0), max(end_index, 600))
    

    lines[0].set_data(range(start_index, end_index), current_speed_data)
    
    # Ear 2
    current_speed_data = np.zeros(600)
    current_speed_data[max(0, 600-(end_index-speed_start_index)):600] = filtered_deg[speed_start_index:end_index]

    axs[1].set_xlim(max(start_index, 0), max(end_index, 600))
    

    lines[1].set_data(range(start_index, end_index), current_speed_data)

    fig.canvas.draw()
    plt.pause(0.001)
    

    plt_img_min = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
    plt_img_min = plt_img_min.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    plt_img_min = cv2.cvtColor(plt_img_min, cv2.COLOR_RGB2BGR)
    plt_img_min = cv2.resize(plt_img_min, (400,800))

    frame = cv2.resize(frame, (1280,800))


    combined_frame = np.hstack([plt_img_min, frame])
    
    # cv2.imshow('1', combined_frame)
    # cv2.waitKey(1)

    out.write(combined_frame)
 
    # if index > 5000:
    #     break


cap.release()
out.release()

In [ ]:
total_angle = np.sum(np.abs(angle_deg))

In [ ]:

cap.release()
out.release()

In [ ]:


abs_angle_deg = np.abs(angle_deg)
bin_size = len(abs_angle_deg) // 100
sums = np.array([np.sum(abs_angle_deg[i*bin_size:(i+1)*bin_size]) for i in range(100)])


plt.figure(figsize=(10, 5))
plt.plot(sums, marker='o', linestyle='-')
plt.title('Sum of degree')
plt.xlabel('Time(60min/100)')
plt.ylabel('Sum of Elements')
plt.grid(True)
plt.show()

In [ ]:
abs_angle_deg = np.abs(angle_deg)
bin_size = len(abs_angle_deg) // 60
sums = np.array([np.sum(abs_angle_deg[i*bin_size:(i+1)*bin_size]) for i in range(60)])


plt.figure(figsize=(10, 5))
plt.plot(sums, marker='o', linestyle='-')
plt.title('Sum of degree')
plt.xlabel('Time(min)')
plt.ylabel('Sum of Elements')
plt.grid(True)
plt.show()